In [0]:
storage_account = dbutils.secrets.get(scope="fraud-kv-scope", key="adls-storage-account-name")
storage_key     = dbutils.secrets.get(scope="fraud-kv-scope", key="adls-storage-account-key")
evh_conn_str    = dbutils.secrets.get(scope="fraud-kv-scope", key="evh-connection-string")
evh_namespace   = "evhb-fraud-pipeline"


In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
bronze_path = f'abfss://bronze@{storage_account}.dfs.core.windows.net/transactions'
silver_path = f'abfss://silver@{storage_account}.dfs.core.windows.net/transactions'
silver_checkpoint = f'abfss://silver@{storage_account}.dfs.core.windows.net/_checkpoints/transactions'

display(dbutils.fs.ls(bronze_path))     

path,name,size,modificationTime
abfss://bronze@[REDACTED].dfs.core.windows.net/transactions/_delta_log/,_delta_log/,0,1775208646000
abfss://bronze@[REDACTED].dfs.core.windows.net/transactions/part-00000-c799d636-cd9e-451a-a35f-e447324ff477.c000.snappy.parquet,part-00000-c799d636-cd9e-451a-a35f-e447324ff477.c000.snappy.parquet,3421,1775208664000
abfss://bronze@[REDACTED].dfs.core.windows.net/transactions/part-00001-a69c6567-0074-4b3a-9c40-6e14ee669311.c000.snappy.parquet,part-00001-a69c6567-0074-4b3a-9c40-6e14ee669311.c000.snappy.parquet,4568,1775208664000


In [0]:
from pyspark.sql.types import *
transaction_schema = StructType([
    StructField('transaction_id', StringType(), True),
    StructField('customer_id', StringType(), True), 
    StructField('customer_name', StringType(), True),
    StructField('account_age_days', IntegerType(), True), 
    StructField('avg_spend', FloatType(), True),
    StructField("spend_std", FloatType(),  True),
    StructField("amount", DoubleType(),  True),
    StructField("merchant", StringType(),  True),
    StructField("location", StringType(),  True),
    StructField("is_foreign", BooleanType(), True),
    StructField("txn_type", StringType(),  True),
    StructField("timestamp", StringType(),  True)

])
print('schame is Defined')

schame is Defined


In [0]:
from pyspark.sql.functions import (
    from_json, col, log1p, when, hour, to_timestamp,
    udf, lit
)

SUSPICIOUS_MERCHANTS = [
    "CASINO", "CRYPTO_EXCHANGE", "UNKNOWN",
    "OFFSHORE_BETTING", "DARK_MARKET"
]

# TXN type encoding — must match training encoding
TXN_TYPE_MAP = {
    "UPI": 0, "NEFT": 1, "IMPS": 2,
    "ATM": 3, "POS": 4, "online": 5
}

@udf(IntegerType())
def encode_txn_type(txn_type):
    return TXN_TYPE_MAP.get(txn_type, -1)

In [0]:
# Read Bronze as stream
bronze_stream = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

# Parse raw JSON → structured columns
parsed = (
    bronze_stream
    .withColumn("txn", from_json(col("raw_json"), transaction_schema))
    .select(
        "txn.*",                        
        "event_hub_timestamp",
        "partition",
        "offset"
    )
)
from pyspark.sql.functions import round as spark_round

featured = (
    parsed
    .withColumn("avg_spend",   spark_round(col("avg_spend"),   2))
    .withColumn("spend_std",   spark_round(col("spend_std"),   2))
    .withColumn("amount",      spark_round(col("amount"),      2))

    # Feature engineering
    .withColumn("amount_log",             spark_round(log1p(col("amount")), 4))
    .withColumn("amount_vs_avg_ratio",    spark_round(col("amount") / col("avg_spend"), 4))
    .withColumn("is_high_amount",         when(col("amount") > 50000, 1).otherwise(0))
    .withColumn("amount_zscore",          spark_round((col("amount") - col("avg_spend")) / col("spend_std"), 4))
    .withColumn("hour_of_day",            hour(to_timestamp(col("timestamp"))))
    .withColumn("is_night",               when((col("hour_of_day") >= 23) | (col("hour_of_day") <= 4), 1).otherwise(0))
    .withColumn("is_foreign_int",         col("is_foreign").cast(IntegerType()))
    .withColumn("is_suspicious_merchant", when(col("merchant").isin(SUSPICIOUS_MERCHANTS), 1).otherwise(0))
    .withColumn("is_new_account",         when(col("account_age_days") < 180, 1).otherwise(0))
    .withColumn("txn_type_encoded",
        when(col("txn_type") == "UPI",    0)
        .when(col("txn_type") == "NEFT",  1)
        .when(col("txn_type") == "IMPS",  2)
        .when(col("txn_type") == "ATM",   3)
        .when(col("txn_type") == "POS",   4)
        .when(col("txn_type") == "online",5)
        .otherwise(-1)
    )
)
print("✅ Feature engineering defined")
featured.printSchema()

✅ Feature engineering defined
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |-- avg_spend: float (nullable = true)
 |-- spend_std: float (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant: string (nullable = true)
 |-- location: string (nullable = true)
 |-- is_foreign: boolean (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- event_hub_timestamp: timestamp (nullable = true)
 |-- partition: string (nullable = true)
 |-- offset: string (nullable = true)
 |-- amount_log: double (nullable = true)
 |-- amount_vs_avg_ratio: double (nullable = true)
 |-- is_high_amount: integer (nullable = false)
 |-- amount_zscore: double (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- is_night: integer (nullable = false)
 |-- is_foreign_int: integer (nullable = true)
 |-- is_

In [0]:
query = (
    featured.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint)
    .option("mergeSchema", "true")
    .trigger(processingTime="10 seconds")
    .start(silver_path)
)

print(f"✅ Silver stream started")
print(f"Status: {query.status}")

✅ Silver stream started
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
display(spark.read.format("delta").load(silver_path))

transaction_id,customer_id,customer_name,account_age_days,avg_spend,spend_std,amount,merchant,location,is_foreign,txn_type,timestamp,event_hub_timestamp,partition,offset,amount_log,amount_vs_avg_ratio,is_high_amount,amount_zscore,hour_of_day,is_night,is_foreign_int,is_suspicious_merchant,is_new_account,txn_type_encoded
001657e2-c79b-4fd0-9f2f-29aa9cc75250,CUST0352,Kabir Bhat,2085,933.94,879.45,2610.17,BigBazaar,Mumbai,false,POS,2026-04-03T07:26:07.091938,2026-04-03T07:26:07.15Z,1,0,7.8676,2.7948,0,1.906,7,0,0,0,0,4
17e6ae1e-d816-4d36-a427-86931a614922,CUST0235,Gokul Sood,3193,5166.39,3013.62,3479.44,Ola,Coimbatore,false,NEFT,2026-04-03T07:26:08.332218,2026-04-03T07:26:07.353Z,1,480,8.1549,0.6735,0,-0.5598,7,0,0,0,0,1
4408a76f-22fc-44b9-8955-21bb918777e8,CUST0339,Mahika Mannan,2401,2182.49,3174.81,5224.65,Blinkit,Mumbai,false,online,2026-04-03T07:26:08.833637,2026-04-03T07:26:07.853Z,1,960,8.5613,2.3939,0,0.9582,7,0,0,0,0,5
e614f153-a997-4f45-9d7b-3297a5096d97,CUST0373,Nirvi Shere,2231,2107.72,2805.89,13909.24,UNKNOWN,Cayman Islands,true,UPI,2026-04-03T01:26:09.335878,2026-04-03T07:26:08.353Z,1,1448,9.5404,6.5992,0,4.206,1,1,1,1,0,0
6ac669f4-cda0-4fef-a52d-9b12615fc79f,CUST0371,Adah Ahluwalia,3233,6390.95,8956.62,6564.1,Flipkart,Chennai,false,online,2026-04-03T07:26:09.838745,2026-04-03T07:26:08.853Z,1,1936,8.7895,1.0271,0,0.0193,7,0,0,0,0,5
46b195bf-8eba-4c95-834b-288648a68754,CUST0214,Ela Balay,2670,11209.6,8735.02,17503.22,DMart,Pune,false,UPI,2026-04-03T07:26:10.340889,2026-04-03T07:26:09.353Z,1,2424,9.7702,1.5614,0,0.7205,7,0,0,0,0,0
c077a082-5dce-47fa-b50f-6258b1839d98,CUST0210,Amani Dhawan,2051,12212.56,14213.5,2552.12,BookMyShow,Kolkata,false,ATM,2026-04-03T07:26:10.842594,2026-04-03T07:26:09.853Z,1,2904,7.8451,0.209,0,-0.6797,7,0,0,0,0,3
eee4c8d6-0bb7-4dc9-ae05-5e11eded206c,CUST0118,Amira Vig,3539,11857.29,10877.89,10265.8,Blinkit,Surat,false,NEFT,2026-04-03T07:26:07.830222,2026-04-03T07:26:07.085Z,0,0,9.2367,0.8658,0,-0.1463,7,0,0,0,0,1
